# Durian Disease Classification — Full Run on Google Colab

End-to-end run of the senior-project pipeline (EDA → train 3 models → evaluate + calibration + abstain threshold → Grad-CAM → OOD → compare).

**Before you start:** set a GPU runtime — *Runtime ▸ Change runtime type ▸ Hardware accelerator ▸ GPU (T4 is fine)*. Then run the cells top to bottom.

Every methodological choice is documented in `reports/DECISION_LOG.md`; results land in `reports/RESULTS.md` and `outputs/`.


## 0. Check the GPU


In [ ]:
!nvidia-smi -L || echo 'No GPU — enable one via Runtime > Change runtime type.'
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable a GPU runtime before continuing.'


## 1. Get the code

Clones the project branch. If the repo is **private**, create a GitHub Personal Access Token and set `GH_TOKEN` below (or upload the project folder to `/content/Senior-Project` instead of cloning).


In [ ]:
import os
REPO_USER = 'Punpunkiki'
REPO_NAME = 'Senior-Project'
BRANCH    = 'claude/durian-disease-classifier-aypmn0'
GH_TOKEN  = ''  # <- only needed for a PRIVATE repo (a GitHub PAT)

url = f'https://github.com/{REPO_USER}/{REPO_NAME}.git'
if GH_TOKEN:
    url = f'https://{GH_TOKEN}@github.com/{REPO_USER}/{REPO_NAME}.git'
target = f'/content/{REPO_NAME}'
if not os.path.isdir(target):
    get_ipython().system(f'git clone --branch {BRANCH} --single-branch {url} {target}')
%cd /content/Senior-Project
get_ipython().system('git log --oneline -1')


## 2. Install dependencies

Colab already ships a CUDA build of `torch`/`torchvision`, so we install only the extras (installing the pinned `torch` from `requirements.txt` could downgrade and break CUDA). If you hit a `timm`/`torch` mismatch, restart the runtime and re-run.


In [ ]:
!pip install -q timm imagehash grad-cam tabulate pyyaml seaborn
import timm, imagehash, pytorch_grad_cam, tabulate, yaml, seaborn
print('extras installed | timm', timm.__version__)


## 3. Data

`./data/` must end up with **12 class sub-folders**: the 10 Mendeley disease folders + a `Healthy/` and a `not_durian/` folder you add yourself (save as JPG; see [DL-CLASSES] for sources). The quarantined covariate-shift OOD set (same diseases, web/social) goes in `./data_ood/` (eval only — never trained on), separate from the not_durian training negatives. Pick **one** option below for the primary data.


### Option A — download from Mendeley (open internet)
Downloads the primary dataset (~GB) straight into `./data/`. Skip if you use Option B.


In [ ]:
# Uncomment to use:
# !python -m src.download_data --yes --dest ./data


### Option B — mount Google Drive (recommended if you already have the data)
Point `DATA_SRC` / `OOD_SRC` at folders in your Drive that each contain the 10 class sub-folders. We symlink them so nothing is copied.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# EDIT these two paths to where YOUR data lives in Drive:
DATA_SRC = '/content/drive/MyDrive/durian/data'        # 10 class folders (Mendeley)
OOD_SRC  = '/content/drive/MyDrive/durian/data_ood'    # 10 class folders (web/social)

import os, shutil
for src_path, link in [(DATA_SRC, 'data'), (OOD_SRC, 'data_ood')]:
    if os.path.isdir(src_path):
        if os.path.islink(link):
            os.unlink(link)
        elif os.path.isdir(link):
            shutil.rmtree(link)
        os.symlink(src_path, link)
        print('linked', link, '->', src_path)
    else:
        print('NOT FOUND (edit the path):', src_path)


### Verify the layout & real per-class counts


In [ ]:
import sys; sys.path.insert(0, os.getcwd())
from src.utils import load_config
from src import data as D
cfg = load_config('config.yaml')
df = D.discover_images(cfg)   # prints the TRUTH (counts, format, imbalance)
df['label'].value_counts()


## 4. (Optional) Colab config tweaks

Writes a `config_colab.yaml` copy (the original stays intact, comments and all) and points every later command at it via `$CFG`. Use this for a quick first end-to-end pass (fewer epochs), then re-run with the full config for your final numbers. **Skip this cell to use the full default config.**


In [ ]:
import yaml, os
cfg_full = yaml.safe_load(open('config.yaml'))
# --- quick-pass overrides (edit as you like) ---
cfg_full['train']['epochs']      = 15      # full default is 50
cfg_full['train']['batch_size']  = 32      # raise to 64 if VRAM allows
cfg_full['train']['num_workers'] = 2       # Colab has ~2 CPU cores
yaml.safe_dump(cfg_full, open('config_colab.yaml', 'w'), sort_keys=False)
os.environ['CFG'] = 'config_colab.yaml'
print('using', os.environ['CFG'], '| epochs =', cfg_full['train']['epochs'])


In [ ]:
# Default config selector (run this if you SKIPPED the tweak cell above).
import os
os.environ.setdefault('CFG', 'config.yaml')
print('config in use:', os.environ['CFG'])


## 5. (Optional) 30-second smoke test
Random-weights forward/backward + split/metric checks — confirms the wiring before the long run. Produces no real metrics.


In [ ]:
!python scripts/smoke_test.py


## Phase 1 — EDA
Verifies counts/resolution/colour stats and detects near-duplicates (leakage risk). Decision Logs: [DL-FORMAT] [DL-RESIZE] [DL-NORM] [DL-GROUP].


In [ ]:
!python -m src.eda --config $CFG


In [ ]:
import json, glob
from IPython.display import Image, display
def show(path, width=520):
    import os
    if os.path.exists(path): display(Image(path, width=width))
    else: print('missing:', path)
for p in ['outputs/eda/class_counts.png','outputs/eda/resolution_scatter.png',
          'outputs/eda/brightness_by_class.png']:
    show(p)
print(json.dumps(json.load(open('outputs/eda/duplicates.json')), indent=2))


## Phases 2–5 — Split + train all 3 models
Group-aware stratified split (saved to `outputs/splits.json` once), then ImageNet-pretrained fine-tuning with LLRD + warmup/cosine + early stop on val macro-F1. This is the long cell (minutes–tens of minutes on a T4).


In [ ]:
!python -m src.train --all --config $CFG


In [ ]:
from src.utils import load_config
cfg = load_config(os.environ['CFG'])
for m in cfg['models']:
    print('===', m['name'], '==='); show(f"outputs/{m['name']}/curves.png", width=760)


## Phases 6, 7, 9 — Evaluate, calibrate, abstain threshold, OOD gap
Metrics + confusion matrices + reliability/ECE + temperature scaling + confidence threshold (from the precision/coverage curve) + the OOD set.


In [ ]:
!python -m src.evaluate --all --config $CFG --ood


In [ ]:
import json
for m in cfg['models']:
    name = m['name']; print('\n========== ', name, ' ==========')
    mj = json.load(open(f'outputs/{name}/metrics.json'))
    print(f"ID  acc={mj['accuracy']:.4f}  macro-F1={mj['macro_f1']:.4f}  "
          f"ECE {mj['ece_raw']:.3f}->{mj['ece_scaled']:.3f} (T={mj['temperature']:.2f})")
    if 'ood' in mj:
        print(f"OOD macro-F1={mj['ood']['macro_f1']:.4f}  "
              f"ID-OOD gap={mj['ood'].get('id_minus_ood_macro_f1'):.4f}")
    ab = mj.get('abstain', {})
    print(f"abstain tau={ab.get('threshold')}  coverage={ab.get('test_coverage')}  "
          f"selective-acc={ab.get('test_selective_accuracy')}")
    for p in [f'outputs/{name}/confusion_matrix.png',
              f'outputs/{name}/confusion_matrix_ood.png',
              f'outputs/{name}/reliability.png',
              f'outputs/{name}/threshold_tradeoff.png']:
        show(p)


## Phase 8 — Interpretability (the central question)
Grad-CAM (CNN + Swin) on correct/incorrect predictions, plus the occlusion test: `cam_over_random_ratio` > 1 means the model relies on the highlighted (ideally lesion) region, not the background.


In [ ]:
!python -m src.interpret --all --config $CFG


In [ ]:
import json, glob
for m in cfg['models']:
    name = m['name']; print('\n==========', name, '==========')
    try:
        print(json.dumps(json.load(open(f'outputs/{name}/occlusion_summary.json')), indent=2))
    except FileNotFoundError:
        pass
    for p in sorted(glob.glob(f'outputs/{name}/gradcam/*.png'))[:8]:
        show(p, width=300)


## Phase 10 — Cross-model comparison & final pick
Single table across ID/OOD macro-F1, ECE, params, and latency. Final selection rule ([DL-FINAL]): prioritise OOD robustness + calibration, then size/latency for edge deployment.


In [ ]:
!python -m src.compare --all --config $CFG


In [ ]:
import pandas as pd
df_cmp = pd.read_csv('outputs/comparison.csv')
show('outputs/comparison.png', width=900)
df_cmp


## Save results back to Drive / download
Zips `outputs/` + the filled reports so you keep them after the runtime ends.


In [ ]:
!zip -qr durian_outputs.zip outputs reports
import os
if os.path.isdir('/content/drive/MyDrive'):
    !cp durian_outputs.zip /content/drive/MyDrive/durian_outputs.zip
    print('saved to Drive: MyDrive/durian_outputs.zip')
try:
    from google.colab import files; files.download('durian_outputs.zip')
except Exception as e:
    print('manual download from the Files panel:', e)


## Next steps
- Fill `reports/RESULTS.md` `[[ ]]` slots from the cells above (the reasoning   in `reports/DECISION_LOG.md` is already complete).
- For final thesis numbers, re-run with the **full** `config.yaml` (skip the   Phase-4 tweak cell) so `epochs=50`.
- Background-reliance ablation: set `data.image_format: png` in the config and   re-run train+evaluate to compare against the JPEG model ([DL-FORMAT]).
